# NEXUS Training Notebook — Google Colab

**Run all cells top-to-bottom.**

| Cell | What it does |
|---|---|
| 1 | Clone / pull the repo |
| 2 | Install dependencies + download ATTNSOM dataset |
| 3 | Environment diagnostic (GPU / RAM) |
| 4 | Single-molecule smoke test (fast forward pass) |
| 5 | 16-molecule training run — inline Python, Colab-safe settings |

All memory-sensitive parameters (quantum grid resolution, chunk size, torch.compile, dynamics mode)
are set directly in Cell 5 — no dependency on `train.py` flags.

In [ ]:
import os, subprocess

REPO = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=True)

os.environ['NEXUS_COLAB_GPU_PROFILE'] = 'ultra_vram'
os.chdir(REPO_DIR)
print('repo:', os.getcwd())
print('gpu_profile:', os.environ['NEXUS_COLAB_GPU_PROFILE'])

In [ ]:
!bash scripts/setup_colab_nexus.sh /content/enzyme_Software

In [ ]:
import torch, psutil

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {props.name}")
    print(f"VRAM : {props.total_memory / 1024**3:.1f} GB")
else:
    print("GPU  : not available — running on CPU (will be slower)")

ram = psutil.virtual_memory()
print(f"RAM  : {ram.total / 1024**3:.1f} GB total,  {ram.available / 1024**3:.1f} GB free")
print(f"Torch: {torch.__version__}")

In [ ]:
!export PYTORCH_ALLOC_CONF=expandable_segments:True && \
python scripts/colab_smoke_test.py \
  --sdf data/ATTNSOM/cyp_dataset/3A4.sdf \
  --gpu-profile ultra_vram \
  --sample-index -1 \
  --steps 1 \
  --forward-only

In [ ]:
# Training code lives in scripts/colab_train.py so git pull always updates it.
exec(open("/content/enzyme_Software/scripts/colab_train.py").read())